# 07 — Delayed Recognition Analysis (LFCR + LFC)

Delayed-recognition extension for the journal-level citation-depth study.

IMPORTANT DESIGN PRINCIPLES
---------------------------
1. Script 07 is downstream of script 06.
   It only uses milestones already initialized by 06 and reads the frozen
   clean dataset, rare-citation summary, and sampling plan saved for the
   selected milestone.
2. Milestone filtering is cumulative and must match the exact eligible file
   list used by 06.
3. Early impact is the original 3-year citation_bin from script 01/06.
4. Later impact is reconstructed from total_citations using the same cohort
   structure (journal-year or journal-period) and rank-based tertile logic.
5. Primary delayed-recognition outcome:
      early Low -> later High
   versus
      early Low -> later Low
   Later Mid is excluded from the binary delayed-recognition models.
6. Two focal citation-depth operationalizations are carried forward:
      M4: continuous LFCR (intensity)
      M5: binary LFC presence (presence of at least one LFC pattern)
   They are modeled separately because LFC is structurally derived from LFCR.
7. In the adjusted delayed-recognition models (M3/M4/M5), the exact early
   3-year citation count (`citations`) is included as a baseline control.
   This asks whether LFCR/LFC relates to later recognition beyond how many
   citations an early-Low paper had already accumulated inside the 3-year window.
8. A4 is an exploratory OOF early-impact follow-up. Because the original
   early citation_bin is derived from 3-year citations, `citations` is
   intentionally EXCLUDED from A4 predictors to avoid target leakage.
9. Two prespecified robustness checks are included:
      (a) post-3-year citation-gain sensitivity, which removes the mechanical
          contribution of early citations from the later-impact quantity;
      (b) exclusion of the most recent publication cohort, which gives the
          remaining papers a longer post-3-year observation window.
10. This is delayed recognition, not a strict Sleeping Beauty analysis, because
    annual citation trajectories are not available.

## 1. SETUP

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd

from scipy.stats import chi2, fisher_exact
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
)

import matplotlib.pyplot as plt
import seaborn as sns

try:
    import shap
    HAS_SHAP = True
except Exception as e:
    HAS_SHAP = False
    print("SHAP is not available; SHAP analyses will be skipped:", e)

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

warnings.filterwarnings("ignore", category=RuntimeWarning)

BASE_DIR = '/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER'
MILESTONES_BASE_DIR = os.path.join(BASE_DIR, "milestones_journal_level")

RANDOM_STATE = 42
TEST_SIZE = 0.20
MIN_MODEL_N = 50
OOF_MAX_SPLITS = 5

GROUP_COL = "journal"
CONTINUOUS_FEATURES = {
    "less_frequent_citation_ratio",
    "citations",  # exact early 3-year citation count; baseline control in delayed-recognition models
    "references",
    "year",
}

# =========================
# PUBLICATION-READY FIGURE SETTINGS
# Matched to script 06 for visual consistency.
# =========================
FIGSIZE_MAIN = (6, 4)
FIGSIZE_WIDE = (9, 5)
FIG_DPI = 300
FIG_FONT_SIZE = 11
BAR_COLOR = "#5975A4"

# Main SHAP figures display substantive predictors only. Journal fixed effects
# (and availability in the full-data models) remain in the fitted model but are
# intentionally omitted from the publication-facing figures.
PLOT_FOCAL_FEATURES_M4 = [
    "less_frequent_citation_ratio",
    "citations",
    "references",
    "year",
]
PLOT_FOCAL_FEATURES_M5 = [
    "has_less_frequent_citation",
    "citations",
    "references",
    "year",
]

# Fixed axes keep M4/M5 and mil1-mil4 directly comparable, following script 06.
SHAP_SUMMARY_XLIM = (-0.6, 3.8)
SHAP_SUMMARY_XTICKS = [-0.5, 0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5]
SHAP_BAR_XLIM = (0, 0.50)
SHAP_BAR_XTICKS = np.arange(0, 0.501, 0.05)

sns.set_theme(style="whitegrid", context="paper")

plt.rcParams.update({
    "font.size": FIG_FONT_SIZE,
    "axes.titlesize": FIG_FONT_SIZE,
    "axes.labelsize": FIG_FONT_SIZE,
    "xtick.labelsize": FIG_FONT_SIZE,
    "ytick.labelsize": FIG_FONT_SIZE,
    "legend.fontsize": FIG_FONT_SIZE,
    "figure.titlesize": FIG_FONT_SIZE,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def apply_figure_fontsize(fig, fontsize=FIG_FONT_SIZE):
    """Force visible Matplotlib/Seaborn/SHAP text to one publication size."""
    for ax in fig.axes:
        ax.title.set_fontsize(fontsize)
        ax.xaxis.label.set_fontsize(fontsize)
        ax.yaxis.label.set_fontsize(fontsize)
        ax.tick_params(axis="both", labelsize=fontsize)

        legend = ax.get_legend()
        if legend is not None:
            for txt in legend.get_texts():
                txt.set_fontsize(fontsize)
            if legend.get_title() is not None:
                legend.get_title().set_fontsize(fontsize)

        for txt in ax.texts:
            txt.set_fontsize(fontsize)

    for txt in fig.texts:
        txt.set_fontsize(fontsize)


FEATURE_LABEL_MAP = {
    "less_frequent_citation_ratio": "LFCR",
    "has_less_frequent_citation": "LFC",
    "has_valid_citation_data": "Valid citation data",
    "citations": "3-year citations",
    "references": "References",
    "year": "Year",
}


def clean_feature_name(name):
    name = str(name)
    if name.startswith("journal_"):
        return name[len("journal_"):]
    return FEATURE_LABEL_MAP.get(name, name.replace("_", " "))


def save_figure(base_path):
    plt.savefig(base_path + ".png", dpi=FIG_DPI, bbox_inches="tight")
    plt.savefig(base_path + ".pdf", bbox_inches="tight")


os.makedirs(MILESTONES_BASE_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("MILESTONES_BASE_DIR:", MILESTONES_BASE_DIR)

## 2. MILESTONE SETUP

In [ ]:
def infer_milestone_order(name):
    match = re.match(r"v(\d+)_", str(name).strip())
    if match is None:
        raise ValueError(
            f"Cannot infer milestone order from: {name}. "
            "Expected names like v1_coverage_30pct, v2_coverage_50pct, ..."
        )
    return int(match.group(1))


def milestone_snapshot_paths(milestone_name):
    milestone_dir = os.path.join(MILESTONES_BASE_DIR, milestone_name)
    input_dir = os.path.join(milestone_dir, "input_files")
    return {
        "milestone_dir": milestone_dir,
        "input_dir": input_dir,
        "clean": os.path.join(input_dir, "clean_dataset_original.csv"),
        "rare": os.path.join(input_dir, "rare_citations_summary.csv"),
        "plan": os.path.join(input_dir, "milestone_sampling_plan.csv"),
        "eligible_list": os.path.join(
            input_dir,
            f"{milestone_name}_cumulative_eligible_filelist_used.txt",
        ),
    }


def is_initialized_by_06(milestone_name):
    paths = milestone_snapshot_paths(milestone_name)
    required = [
        paths["clean"],
        paths["rare"],
        paths["plan"],
        paths["eligible_list"],
    ]
    return all(os.path.exists(p) for p in required)


existing_dirs = []
if os.path.isdir(MILESTONES_BASE_DIR):
    for d in os.listdir(MILESTONES_BASE_DIR):
        full = os.path.join(MILESTONES_BASE_DIR, d)
        if not os.path.isdir(full):
            continue
        try:
            infer_milestone_order(d)
        except Exception:
            continue
        existing_dirs.append(d)

initialized_milestones = sorted(
    [m for m in existing_dirs if is_initialized_by_06(m)],
    key=infer_milestone_order,
)

if not initialized_milestones:
    raise FileNotFoundError(
        "No milestone initialized by script 06 was found.\n"
        "Run 06 first for at least one milestone. Script 07 requires the frozen "
        "06 snapshot files and the 06 eligible-file list."
    )

print("\nMilestones initialized by 06 and available to 07:")
for i, name in enumerate(initialized_milestones, start=1):
    print(f"{i}. {name} (order={infer_milestone_order(name)})")

user_input = input(
    "\nSelect milestone by name or number, e.g. v2_coverage_50pct or 2: "
).strip()

if user_input.isdigit():
    idx = int(user_input) - 1
    if idx < 0 or idx >= len(initialized_milestones):
        raise ValueError(f"Invalid milestone choice: {user_input}")
    MILESTONE = initialized_milestones[idx]
elif user_input:
    MILESTONE = user_input
else:
    MILESTONE = initialized_milestones[0]

if MILESTONE not in initialized_milestones:
    raise ValueError(
        f"Milestone '{MILESTONE}' is not a complete 06-initialized milestone.\n"
        f"Available: {initialized_milestones}"
    )

PATHS = milestone_snapshot_paths(MILESTONE)
MILESTONE_DIR = PATHS["milestone_dir"]
INPUT_SNAPSHOT_DIR = PATHS["input_dir"]
OUTPUT_DIR = os.path.join(MILESTONE_DIR, "outputs", "delayed_recognition")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLEAN_DATASET_PATH = PATHS["clean"]
RARE_DATASET_PATH = PATHS["rare"]
SAMPLING_PLAN_PATH = PATHS["plan"]
ELIGIBLE_FILELIST_06_PATH = PATHS["eligible_list"]

print("\nSelected milestone:", MILESTONE)
print("Frozen clean dataset:", CLEAN_DATASET_PATH)
print("Frozen rare summary:", RARE_DATASET_PATH)
print("Frozen sampling plan:", SAMPLING_PLAN_PATH)
print("06 eligible file list:", ELIGIBLE_FILELIST_06_PATH)
print("07 output directory:", OUTPUT_DIR)

## 3. LOAD FROZEN 06 SNAPSHOTS

In [ ]:
base_df = pd.read_csv(CLEAN_DATASET_PATH)
rare_df = pd.read_csv(RARE_DATASET_PATH)
sampling_df = pd.read_csv(SAMPLING_PLAN_PATH)

rename_map = {
    "rare_citation_ratio": "less_frequent_citation_ratio",
    "correct_rare_citation_ratio": "has_valid_citation_data",
    "has_rare": "has_less_frequent_citation",
}

base_df = base_df.rename(
    columns={k: v for k, v in rename_map.items() if k in base_df.columns and v not in base_df.columns}
)
rare_df = rare_df.rename(
    columns={k: v for k, v in rename_map.items() if k in rare_df.columns and v not in rare_df.columns}
)
sampling_df = sampling_df.rename(
    columns={k: v for k, v in rename_map.items() if k in sampling_df.columns and v not in sampling_df.columns}
)

required_clean_cols = [
    "filename",
    "journal",
    "year",
    "citations",
    "total_citations",
    "citation_bin",
    "references",
    "binning_level",
    "publication_period",
]
missing_clean = [c for c in required_clean_cols if c not in base_df.columns]
if missing_clean:
    raise ValueError(f"Missing required columns from frozen clean dataset: {missing_clean}")

required_rare_cols = ["filename", "less_frequent_citation_ratio"]
missing_rare = [c for c in required_rare_cols if c not in rare_df.columns]
if missing_rare:
    raise ValueError(f"Missing required columns from frozen rare summary: {missing_rare}")

required_plan_cols = ["filename", "add_in_milestone"]
missing_plan = [c for c in required_plan_cols if c not in sampling_df.columns]
if missing_plan:
    raise ValueError(f"Missing required columns from frozen sampling plan: {missing_plan}")

for data in [base_df, rare_df, sampling_df]:
    data["filename"] = data["filename"].astype(str).str.strip()

if base_df["filename"].duplicated().any():
    dup = base_df.loc[base_df["filename"].duplicated(keep=False), "filename"].unique()
    raise ValueError(f"Duplicate filenames in frozen clean dataset. Examples: {dup[:10]}")

if rare_df["filename"].duplicated().any():
    dup = rare_df.loc[rare_df["filename"].duplicated(keep=False), "filename"].unique()
    raise ValueError(f"Duplicate filenames in frozen rare summary. Examples: {dup[:10]}")

if sampling_df["filename"].duplicated().any():
    dup = sampling_df.loc[sampling_df["filename"].duplicated(keep=False), "filename"].unique()
    raise ValueError(f"Duplicate filenames in frozen sampling plan. Examples: {dup[:10]}")

if "milestone_order" not in sampling_df.columns:
    sampling_df["milestone_order"] = sampling_df["add_in_milestone"].apply(infer_milestone_order)

sampling_df["milestone_order"] = pd.to_numeric(sampling_df["milestone_order"], errors="coerce")
if sampling_df["milestone_order"].isna().any():
    bad = sampling_df.loc[sampling_df["milestone_order"].isna(), "add_in_milestone"].unique()
    raise ValueError(f"Invalid milestone_order values in frozen sampling plan: {bad}")

if MILESTONE not in set(sampling_df["add_in_milestone"].astype(str)):
    raise ValueError(
        f"Selected milestone '{MILESTONE}' does not exist in its frozen milestone_sampling_plan.csv"
    )

CURRENT_MILESTONE_ORDER = int(
    sampling_df.loc[
        sampling_df["add_in_milestone"].astype(str) == MILESTONE,
        "milestone_order",
    ].iloc[0]
)

print("Current milestone order from frozen plan:", CURRENT_MILESTONE_ORDER)
print("Frozen clean rows:", len(base_df))
print("Frozen rare-summary rows:", len(rare_df))
print("Frozen sampling-plan rows:", len(sampling_df))

## 4. RECONSTRUCT LATER IMPACT ON FULL FROZEN CLEAN SNAPSHOT

In [ ]:
# Preserve deterministic original row order. clean_dataset produced by 01 is restored
# to filename order after original binning, which itself derives from the stable
# journal/year/title/doi ordering used by 01.
base_df = base_df.copy()
base_df["_original_row_order_07"] = np.arange(len(base_df))

for col in ["year", "citations", "total_citations", "references"]:
    base_df[col] = pd.to_numeric(base_df[col], errors="coerce")

base_df["citation_bin"] = base_df["citation_bin"].astype(str).str.strip().str.lower()
base_df["binning_level"] = base_df["binning_level"].astype(str).str.strip()
base_df["publication_period"] = base_df["publication_period"].astype(str).str.strip()

allowed_binning_levels = {"journal_year", "journal_period"}
observed_levels = set(base_df["binning_level"].dropna().unique())
unknown_levels = observed_levels - allowed_binning_levels
if unknown_levels:
    raise ValueError(f"Unknown binning_level values: {sorted(unknown_levels)}")


# Post-3-year citation accumulation is computed before cohort binning so that a
# sensitivity outcome can be built from citation gain AFTER the early window.
base_df["post_3yr_citations"] = base_df["total_citations"] - base_df["citations"]
base_df["negative_post_3yr_flag"] = (base_df["post_3yr_citations"] < 0).astype(int)
base_df["post_3yr_citations"] = base_df["post_3yr_citations"].clip(lower=0)

base_df["post_3yr_share_of_total"] = np.where(
    base_df["total_citations"] > 0,
    base_df["post_3yr_citations"] / base_df["total_citations"],
    np.nan,
)


def assign_tertile_bins(group, value_col, output_col):
    """Exact rank-based qcut logic inherited from script 01."""
    group = group.copy()

    if len(group) < 3 or group[value_col].nunique(dropna=True) < 2:
        group[output_col] = np.nan
        return group

    try:
        ranked = group[value_col].rank(method="first")
        group[output_col] = pd.qcut(
            ranked,
            q=3,
            labels=["low", "mid", "high"],
        )
    except Exception:
        group[output_col] = np.nan

    return group


later_parts = []

journal_year_rows = base_df[base_df["binning_level"] == "journal_year"].copy()
for _, group in journal_year_rows.groupby(["journal", "year"], sort=False):
    group = assign_tertile_bins(
        group,
        value_col="total_citations",
        output_col="later_citation_bin",
    )
    group = assign_tertile_bins(
        group,
        value_col="post_3yr_citations",
        output_col="post_3yr_gain_bin",
    )
    later_parts.append(group)

journal_period_rows = base_df[base_df["binning_level"] == "journal_period"].copy()
for _, group in journal_period_rows.groupby(["journal", "publication_period"], sort=False):
    group = assign_tertile_bins(
        group,
        value_col="total_citations",
        output_col="later_citation_bin",
    )
    group = assign_tertile_bins(
        group,
        value_col="post_3yr_citations",
        output_col="post_3yr_gain_bin",
    )
    later_parts.append(group)

if not later_parts:
    raise ValueError("No rows available for later-impact binning.")

base_df = pd.concat(later_parts, ignore_index=True)
base_df = base_df.sort_values("_original_row_order_07").reset_index(drop=True)
base_df["later_citation_bin"] = base_df["later_citation_bin"].astype("object")
base_df["post_3yr_gain_bin"] = base_df["post_3yr_gain_bin"].astype("object")

later_binning_quality = pd.DataFrame({
    "metric": [
        "N_frozen_clean",
        "N_later_bin_assigned",
        "N_later_bin_missing",
        "N_post_3yr_gain_bin_assigned",
        "N_post_3yr_gain_bin_missing",
        "N_negative_raw_post_3yr_before_clipping",
    ],
    "value": [
        len(base_df),
        int(base_df["later_citation_bin"].notna().sum()),
        int(base_df["later_citation_bin"].isna().sum()),
        int(base_df["post_3yr_gain_bin"].notna().sum()),
        int(base_df["post_3yr_gain_bin"].isna().sum()),
        int(base_df["negative_post_3yr_flag"].sum()),
    ],
})

later_binning_quality.to_csv(
    os.path.join(OUTPUT_DIR, "later_binning_quality.csv"),
    index=False,
)

print("\nLater-impact binning quality:")
display(later_binning_quality)
print("Later citation-bin counts on full frozen snapshot:")
print(base_df["later_citation_bin"].value_counts(dropna=False))
print("Post-3-year citation-gain bin counts on full frozen snapshot:")
print(base_df["post_3yr_gain_bin"].value_counts(dropna=False))

## 5. MERGE LFCR FROM FROZEN RARE SUMMARY

In [ ]:
# Always take LFCR from the frozen rare_citations_summary snapshot, as 06 does.
if "less_frequent_citation_ratio" in base_df.columns:
    base_df = base_df.drop(columns=["less_frequent_citation_ratio"])

rare_for_merge = rare_df[["filename", "less_frequent_citation_ratio"]].copy()
rare_for_merge["less_frequent_citation_ratio"] = pd.to_numeric(
    rare_for_merge["less_frequent_citation_ratio"], errors="coerce"
)

merged_df = base_df.merge(
    rare_for_merge,
    on="filename",
    how="left",
    validate="one_to_one",
)

merged_df["has_valid_citation_data"] = merged_df["less_frequent_citation_ratio"].notna().astype(int)
merged_df["has_less_frequent_citation"] = (
    merged_df["less_frequent_citation_ratio"] > 0
).astype(int)

print("\nMerged frozen dataset shape:", merged_df.shape)
print("LFCR coverage on full frozen snapshot:", merged_df["has_valid_citation_data"].mean())

## 6. CUMULATIVE MILESTONE FILTERING + STRICT 06 CONSISTENCY CHECK

In [ ]:
eligible_plan = sampling_df[
    sampling_df["milestone_order"] <= CURRENT_MILESTONE_ORDER
].copy()

cols_to_add = ["filename", "add_in_milestone", "milestone_order"]
if "milestone_pct" in eligible_plan.columns:
    cols_to_add.append("milestone_pct")

eligible_plan_small = eligible_plan[cols_to_add].drop_duplicates("filename")

before_filter_n = len(merged_df)
df = merged_df.merge(
    eligible_plan_small,
    on="filename",
    how="inner",
    validate="one_to_one",
)
after_filter_n = len(df)

print("\nCumulative milestone filtering applied")
print("Current milestone:", MILESTONE)
print("Rows before filtering:", before_filter_n)
print("Rows after filtering:", after_filter_n)
print("Included add_in_milestone values:")
print(df["add_in_milestone"].value_counts().sort_index())

# Strict consistency against the exact file list saved by 06.
files_06 = set(
    pd.read_csv(
        ELIGIBLE_FILELIST_06_PATH,
        header=None,
        names=["filename"],
        dtype=str,
    )["filename"].astype(str).str.strip()
)
files_07 = set(df["filename"].astype(str).str.strip())

missing_from_07 = sorted(files_06 - files_07)
extra_in_07 = sorted(files_07 - files_06)
consistency_passed = (len(missing_from_07) == 0 and len(extra_in_07) == 0)

consistency_df = pd.DataFrame([{
    "milestone": MILESTONE,
    "N_files_06": len(files_06),
    "N_files_07": len(files_07),
    "N_missing_from_07": len(missing_from_07),
    "N_extra_in_07": len(extra_in_07),
    "consistency_passed": consistency_passed,
    "missing_examples": "; ".join(missing_from_07[:20]),
    "extra_examples": "; ".join(extra_in_07[:20]),
}])

consistency_df.to_csv(
    os.path.join(OUTPUT_DIR, "milestone_consistency_check_06_vs_07.csv"),
    index=False,
)

print("\nMILESTONE CONSISTENCY CHECK")
display(consistency_df)

if not consistency_passed:
    raise RuntimeError(
        "07 eligible-paper set differs from the exact eligible-paper set saved by 06. "
        "Analysis stopped to protect reproducibility."
    )

# Keep a local copy of the exact file list used by 07.
pd.Series(sorted(files_07)).to_csv(
    os.path.join(OUTPUT_DIR, f"{MILESTONE}_07_cumulative_eligible_filelist_used.txt"),
    index=False,
    header=False,
)

## 7. A1 — IMPACT TRANSITIONS / DESCRIPTIVES

In [ ]:
# Explicit transition label. Missing later bin is retained descriptively but excluded
# from binary delayed-recognition modeling.
def make_transition(row):
    early = row["citation_bin"] if pd.notna(row["citation_bin"]) else "missing"
    later = row["later_citation_bin"] if pd.notna(row["later_citation_bin"]) else "missing"
    return f"{early}_to_{later}"


df["impact_transition"] = df.apply(make_transition, axis=1)
df["delayed_recognition_candidate"] = (
    (df["citation_bin"] == "low")
    & (df["later_citation_bin"] == "high")
).astype(int)

df["persistent_low"] = (
    (df["citation_bin"] == "low")
    & (df["later_citation_bin"] == "low")
).astype(int)

# Count table.
transition_counts = pd.crosstab(
    df["citation_bin"],
    df["later_citation_bin"],
    dropna=False,
)
transition_counts.to_csv(
    os.path.join(OUTPUT_DIR, "impact_transition_table_counts.csv")
)

# Row-normalized table.
transition_row_pct = pd.crosstab(
    df["citation_bin"],
    df["later_citation_bin"],
    normalize="index",
    dropna=False,
)
transition_row_pct.to_csv(
    os.path.join(OUTPUT_DIR, "impact_transition_table_row_proportions.csv")
)

transition_long = (
    df.groupby(
        ["citation_bin", "later_citation_bin"],
        dropna=False,
    )
    .size()
    .reset_index(name="N")
)
transition_long.to_csv(
    os.path.join(OUTPUT_DIR, "impact_transition_table_long.csv"),
    index=False,
)

print("\nA1 — Early -> Later impact transition counts")
display(transition_counts)
print("\nA1 — Row proportions")
display(transition_row_pct)

# Main DR modeling population: early Low only, later Low vs High; later Mid is omitted.
dr_all = df[
    (df["citation_bin"] == "low")
    & (df["later_citation_bin"].isin(["low", "high"]))
].copy()
dr_all["label"] = (dr_all["later_citation_bin"] == "high").astype(int)
dr_all["delayed_recognition_label"] = dr_all["label"]

print("\nDelayed-recognition modeling population")
print("N:", len(dr_all))
print(dr_all["label"].value_counts().rename(index={0: "Persistent Low", 1: "Delayed Recognition"}))

# Descriptives by journal. LFC presence rate is included alongside LFCR.
dr_by_journal = (
    dr_all.groupby("journal")
    .agg(
        N=("filename", "count"),
        delayed_N=("label", "sum"),
        delayed_rate=("label", "mean"),
        mean_LFCR=("less_frequent_citation_ratio", "mean"),
        median_LFCR=("less_frequent_citation_ratio", "median"),
        LFC_presence_rate=("has_less_frequent_citation", "mean"),
        mean_early_3yr_citations=("citations", "mean"),
        mean_total_citations=("total_citations", "mean"),
        mean_post_3yr_citations=("post_3yr_citations", "mean"),
    )
    .reset_index()
    .sort_values("N", ascending=False)
)
dr_by_journal.to_csv(
    os.path.join(OUTPUT_DIR, "delayed_recognition_descriptives_by_journal.csv"),
    index=False,
)

# Compact LFCR/LFC descriptives by delayed-recognition outcome.
dr_by_outcome = (
    dr_all.groupby("label")
    .agg(
        N=("filename", "count"),
        mean_LFCR=("less_frequent_citation_ratio", "mean"),
        median_LFCR=("less_frequent_citation_ratio", "median"),
        LFC_N=("has_less_frequent_citation", "sum"),
        LFC_presence_rate=("has_less_frequent_citation", "mean"),
    )
    .reset_index()
)
dr_by_outcome["Outcome"] = dr_by_outcome["label"].map(
    {0: "Persistent Low", 1: "Delayed Recognition"}
)
dr_by_outcome.to_csv(
    os.path.join(OUTPUT_DIR, "delayed_recognition_LFCR_LFC_descriptives_by_outcome.csv"),
    index=False,
)
print("\nLFCR/LFC descriptives by delayed-recognition outcome")
display(dr_by_outcome)

# Save enriched milestone dataset and DR-specific dataset.
df.drop(columns=["_original_row_order_07"], errors="ignore").to_csv(
    os.path.join(OUTPUT_DIR, "delayed_recognition_enriched_milestone_dataset.csv"),
    index=False,
)
dr_all.drop(columns=["_original_row_order_07"], errors="ignore").to_csv(
    os.path.join(OUTPUT_DIR, "delayed_recognition_binary_analysis_dataset.csv"),
    index=False,
)

## 8. VALID-ONLY / FULL DATASETS

In [ ]:
# Mirrors 06.
df_valid = dr_all[dr_all["has_valid_citation_data"] == 1].copy()

df_full = dr_all.copy()
df_full["less_frequent_citation_ratio"] = df_full["less_frequent_citation_ratio"].fillna(0)

print("\nDR valid-only N:", len(df_valid))
print("DR full-dataset N:", len(df_full))
print("DR valid-only class counts:")
print(df_valid["label"].value_counts())
print("DR full class counts:")
print(df_full["label"].value_counts())

## 9. MODEL DEFINITIONS

In [ ]:
# M1/M2 are unadjusted descriptive association models.
# M3/M4/M5 are the adjusted delayed-recognition models.
# IMPORTANT: exact early 3-year citations are included in M3/M4/M5 so that
# LFCR/LFC are tested beyond the paper's citation level at the end of the early window.
experiment_dict_valid = {
    "M1_ratio_only": ["less_frequent_citation_ratio"],
    "M2_presence_only": ["has_less_frequent_citation"],
    "M3_controls_only": ["citations", "references", "year", "journal"],
    "M4_ratio_adjusted": [
        "less_frequent_citation_ratio",
        "citations",
        "references",
        "year",
        "journal",
    ],
    "M5_presence_adjusted": [
        "has_less_frequent_citation",
        "citations",
        "references",
        "year",
        "journal",
    ],
}

experiment_dict_full = {
    "M0_availability_only": ["has_valid_citation_data"],
    "M1_ratio_plus_availability": [
        "less_frequent_citation_ratio",
        "has_valid_citation_data",
    ],
    "M2_presence_plus_availability": [
        "has_less_frequent_citation",
        "has_valid_citation_data",
    ],
    "M3_controls_plus_availability": [
        "has_valid_citation_data",
        "citations",
        "references",
        "year",
        "journal",
    ],
    "M4_ratio_adjusted": [
        "less_frequent_citation_ratio",
        "has_valid_citation_data",
        "citations",
        "references",
        "year",
        "journal",
    ],
    "M5_presence_adjusted": [
        "has_less_frequent_citation",
        "has_valid_citation_data",
        "citations",
        "references",
        "year",
        "journal",
    ],
}

## 10. MODEL HELPERS

In [ ]:
def has_required_columns(data, features, target="label"):
    required = list(dict.fromkeys(features + [target]))
    missing = [c for c in required if c not in data.columns]
    if missing:
        print("Missing columns:", missing)
        return False
    return True


def build_design_matrix(model_data, features):
    X = model_data[features].copy()

    if GROUP_COL in X.columns:
        levels = sorted(X[GROUP_COL].dropna().astype(str).unique())
        X[GROUP_COL] = pd.Categorical(
            X[GROUP_COL].astype(str),
            categories=levels,
            ordered=False,
        )
        X = pd.get_dummies(
            X,
            columns=[GROUP_COL],
            prefix=GROUP_COL,
            drop_first=True,
            dtype=float,
        )

    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    return X.astype(float)


def remove_zero_variance_features(X):
    zero_var = X.columns[X.nunique(dropna=False) <= 1].tolist()
    if zero_var:
        X = X.drop(columns=zero_var)
    return X, zero_var


def scale_predictive_train_test(X_train, X_test):
    X_train_out = X_train.copy().astype(float)
    X_test_out = X_test.copy().astype(float)

    continuous_cols = [c for c in X_train_out.columns if c in CONTINUOUS_FEATURES]
    if continuous_cols:
        scaler = StandardScaler()
        X_train_out.loc[:, continuous_cols] = scaler.fit_transform(
            X_train_out[continuous_cols]
        )
        X_test_out.loc[:, continuous_cols] = scaler.transform(
            X_test_out[continuous_cols]
        )

    return X_train_out, X_test_out


def scale_predictive_full(X):
    X_out = X.copy().astype(float)
    continuous_cols = [c for c in X_out.columns if c in CONTINUOUS_FEATURES]
    if continuous_cols:
        scaler = StandardScaler()
        X_out.loc[:, continuous_cols] = scaler.fit_transform(X_out[continuous_cols])
    return X_out


def safe_logit_fit(y, X_df):
    X_sm = sm.add_constant(X_df, has_constant="add")
    return sm.Logit(
        y.reset_index(drop=True),
        X_sm.reset_index(drop=True),
    ).fit(disp=0)


def compute_metrics(y_true, y_pred, y_score=None):
    out = {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Balanced_Accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
    }

    if y_score is not None and len(np.unique(y_true)) == 2:
        try:
            out["ROC_AUC"] = float(roc_auc_score(y_true, y_score))
        except Exception:
            out["ROC_AUC"] = np.nan
        try:
            out["Average_Precision"] = float(average_precision_score(y_true, y_score))
        except Exception:
            out["Average_Precision"] = np.nan

    return out

## 11. A2 — PREDICTIVE DELAYED-RECOGNITION MODELS

In [ ]:
def run_predictive_analysis(data, scenario_name, experiment_dict):
    scenario_dir = os.path.join(OUTPUT_DIR, scenario_name)
    os.makedirs(scenario_dir, exist_ok=True)

    rows = []

    print(f"\n\n######## A2 PREDICTIVE DR: {scenario_name} ########")

    for model_name, features in experiment_dict.items():
        if not has_required_columns(data, features):
            continue

        model_data = data.dropna(subset=features + ["label"]).copy()
        if len(model_data) < MIN_MODEL_N or model_data["label"].nunique() < 2:
            print(f"Skipping {model_name}: insufficient data/classes")
            continue

        min_class_n = int(model_data["label"].value_counts().min())
        if min_class_n < 2:
            print(f"Skipping {model_name}: minority class < 2")
            continue

        X = build_design_matrix(model_data, features)
        y = model_data["label"].astype(int).copy()
        X, zero_var = remove_zero_variance_features(X)
        if X.shape[1] == 0:
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=y,
        )

        X_train, train_zero = remove_zero_variance_features(X_train)
        X_test = X_test[X_train.columns].copy()

        X_train_model, X_test_model = scale_predictive_train_test(X_train, X_test)

        model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
        model.fit(X_train_model, y_train)

        y_pred = model.predict(X_test_model)
        y_score = model.predict_proba(X_test_model)[:, 1]
        metrics = compute_metrics(y_test, y_pred, y_score)

        row = {
            "Scenario": scenario_name,
            "Model": model_name,
            "N": len(model_data),
            "Positive_N": int(y.sum()),
            "Positive_Rate": float(y.mean()),
            "Features": ", ".join(X_train_model.columns),
        }
        row.update(metrics)
        rows.append(row)

    results_df = pd.DataFrame(rows)
    results_df.to_csv(
        os.path.join(scenario_dir, f"dr_predictive_performance_{scenario_name}.csv"),
        index=False,
    )
    display(results_df)
    return results_df

## 12. A2 — FULL-SAMPLE INFERENTIAL MODELS

In [ ]:
def run_inferential_analysis(data, scenario_name, experiment_dict):
    scenario_dir = os.path.join(OUTPUT_DIR, scenario_name)
    os.makedirs(scenario_dir, exist_ok=True)

    rows = []
    print(f"\n\n######## A2 FULL-SAMPLE INFERENCE: {scenario_name} ########")

    for model_name, features in experiment_dict.items():
        if not has_required_columns(data, features):
            continue

        model_data = data.dropna(subset=features + ["label"]).copy()
        if len(model_data) < MIN_MODEL_N or model_data["label"].nunique() < 2:
            print(f"Skipping {model_name}: insufficient data/classes")
            continue

        X = build_design_matrix(model_data, features)
        y = model_data["label"].astype(int).copy()
        X, zero_var = remove_zero_variance_features(X)
        if X.shape[1] == 0:
            continue

        # Inferential convention: center year only for numerical stability; keep LFCR,
        # early 3-year citations, and references in their natural units.
        if "year" in X.columns:
            X["year"] = X["year"] - X["year"].mean()

        X_sm = sm.add_constant(X, has_constant="add")

        try:
            fitted = sm.Logit(y, X_sm).fit(disp=0)
            conf = fitted.conf_int()

            for feature in fitted.params.index:
                beta = float(fitted.params[feature])
                ci_low = float(conf.loc[feature, 0])
                ci_high = float(conf.loc[feature, 1])

                row = {
                    "Scenario": scenario_name,
                    "Model": model_name,
                    "Feature": feature,
                    "N": len(model_data),
                    "Positive_N": int(y.sum()),
                    "Positive_Rate": float(y.mean()),
                    "Coefficient": beta,
                    "Std_Error": float(fitted.bse[feature]),
                    "Odds_Ratio": float(np.exp(beta)),
                    "OR_CI95_Lower": float(np.exp(ci_low)),
                    "OR_CI95_Upper": float(np.exp(ci_high)),
                    "p-value": float(fitted.pvalues[feature]),
                    "Pseudo_R2": float(fitted.prsquared),
                    "AIC": float(fitted.aic),
                    "BIC": float(fitted.bic),
                }

                if feature == "less_frequent_citation_ratio":
                    row["OR_per_0.10_ratio_increase"] = float(np.exp(beta * 0.10))
                    row["OR_per_0.10_CI95_Lower"] = float(np.exp(ci_low * 0.10))
                    row["OR_per_0.10_CI95_Upper"] = float(np.exp(ci_high * 0.10))

                rows.append(row)

        except Exception as e:
            print(f"Statsmodels failed for {scenario_name}/{model_name}: {e}")

    inferential_df = pd.DataFrame(rows)
    inferential_df.to_csv(
        os.path.join(scenario_dir, f"dr_logistic_inferential_fullsample_{scenario_name}.csv"),
        index=False,
    )
    display(inferential_df)
    return inferential_df

## 13. A3 — NESTED LR TESTS: M3 VS M4 (LFCR) AND M3 VS M5 (LFC)

In [ ]:
def run_incremental_lr_tests(data, scenario_name, experiment_dict):
    """
    A3 nested likelihood-ratio tests for the two prespecified citation-depth
    operationalizations:

      M3 -> M4 : add continuous LFCR
      M3 -> M5 : add binary LFC presence

    M3 already controls for the exact early 3-year citation count, references,
    publication year, and journal (plus availability in the full-data scenario).
    The reduced and full models in each comparison use exactly the same rows.
    Raw p-values are retained. A BH/FDR-adjusted p-value is also reported across
    the two focal incremental tests within each scenario.
    """
    scenario_dir = os.path.join(OUTPUT_DIR, scenario_name)
    os.makedirs(scenario_dir, exist_ok=True)

    reduced_name = (
        "M3_controls_only"
        if scenario_name == "valid_only"
        else "M3_controls_plus_availability"
    )

    comparisons = [
        {
            "full_name": "M4_ratio_adjusted",
            "added_term": "less_frequent_citation_ratio",
            "construct": "LFCR",
        },
        {
            "full_name": "M5_presence_adjusted",
            "added_term": "has_less_frequent_citation",
            "construct": "LFC",
        },
    ]

    rows = []

    for spec in comparisons:
        full_name = spec["full_name"]
        added_term = spec["added_term"]
        construct = spec["construct"]

        reduced_features = experiment_dict[reduced_name]
        full_features = experiment_dict[full_name]

        # Exact same rows for the two nested models in THIS comparison.
        model_data = data.dropna(subset=full_features + ["label"]).copy()

        if len(model_data) < MIN_MODEL_N or model_data["label"].nunique() < 2:
            print(
                f"Skipping LR test {reduced_name} vs {full_name} "
                f"for {scenario_name}: insufficient data/classes"
            )
            continue

        y = model_data["label"].astype(int).copy()
        X_reduced = build_design_matrix(model_data, reduced_features)
        X_full = build_design_matrix(model_data, full_features)

        X_reduced, _ = remove_zero_variance_features(X_reduced)
        X_full, _ = remove_zero_variance_features(X_full)

        # Same centering convention as the full-sample inferential models.
        if "year" in X_reduced.columns:
            year_mean = X_reduced["year"].mean()
            X_reduced["year"] = X_reduced["year"] - year_mean
            if "year" in X_full.columns:
                X_full["year"] = X_full["year"] - year_mean

        try:
            reduced_fit = sm.Logit(
                y,
                sm.add_constant(X_reduced, has_constant="add"),
            ).fit(disp=0)

            full_fit = sm.Logit(
                y,
                sm.add_constant(X_full, has_constant="add"),
            ).fit(disp=0)

            lr_stat = 2.0 * (full_fit.llf - reduced_fit.llf)
            df_diff = int(full_fit.df_model - reduced_fit.df_model)
            p_value = float(chi2.sf(lr_stat, df_diff)) if df_diff > 0 else np.nan

            rows.append({
                "Scenario": scenario_name,
                "Reduced_Model": reduced_name,
                "Full_Model": full_name,
                "Construct": construct,
                "Added_Term": added_term,
                "N": len(model_data),
                "Positive_N": int(y.sum()),
                "LR_statistic": float(lr_stat),
                "df": df_diff,
                "p-value": p_value,
                "Reduced_Pseudo_R2": float(reduced_fit.prsquared),
                "Full_Pseudo_R2": float(full_fit.prsquared),
                "Delta_Pseudo_R2": float(full_fit.prsquared - reduced_fit.prsquared),
                "Reduced_AIC": float(reduced_fit.aic),
                "Full_AIC": float(full_fit.aic),
                "Delta_AIC_Full_minus_Reduced": float(full_fit.aic - reduced_fit.aic),
                "Interpretation": (
                    f"Significant => {construct} adds explanatory information beyond controls"
                ),
            })

        except Exception as e:
            print(
                f"LR test failed for {scenario_name}: "
                f"{reduced_name} vs {full_name}: {e}"
            )

    out = pd.DataFrame(rows)

    # BH/FDR across the two focal A3 incremental tests within scenario.
    if not out.empty and "p-value" in out.columns:
        valid_mask = out["p-value"].notna()
        if valid_mask.sum() > 0:
            reject, p_bh, _, _ = multipletests(
                out.loc[valid_mask, "p-value"].astype(float),
                alpha=0.05,
                method="fdr_bh",
            )
            out.loc[valid_mask, "p-value_BH"] = p_bh
            out.loc[valid_mask, "significant_BH_0.05"] = reject
            out["BH_family"] = (
                "M3->M4 LFCR and M3->M5 LFC incremental LR tests within scenario"
            )

    # Main combined file.
    out.to_csv(
        os.path.join(
            scenario_dir,
            f"dr_incremental_LR_M3_vs_M4_M5_{scenario_name}.csv",
        ),
        index=False,
    )

    # Compatibility / convenient separate files.
    if not out.empty:
        out[out["Full_Model"] == "M4_ratio_adjusted"].to_csv(
            os.path.join(
                scenario_dir,
                f"dr_incremental_LR_M3_vs_M4_{scenario_name}.csv",
            ),
            index=False,
        )
        out[out["Full_Model"] == "M5_presence_adjusted"].to_csv(
            os.path.join(
                scenario_dir,
                f"dr_incremental_LR_M3_vs_M5_{scenario_name}.csv",
            ),
            index=False,
        )

    display(out)
    return out


def build_incremental_predictive_comparison(performance_df, scenario_name):
    """
    Compare predictive performance of the controls-only model (M3) against
    M4 (controls + LFCR) and M5 (controls + binary LFC presence).
    """
    if performance_df.empty:
        return pd.DataFrame()

    reduced_name = (
        "M3_controls_only"
        if scenario_name == "valid_only"
        else "M3_controls_plus_availability"
    )

    comparisons = [
        ("M4_ratio_adjusted", "less_frequent_citation_ratio", "LFCR"),
        ("M5_presence_adjusted", "has_less_frequent_citation", "LFC"),
    ]

    metrics = [
        "Accuracy",
        "Balanced_Accuracy",
        "F1",
        "Precision",
        "Recall",
        "ROC_AUC",
        "Average_Precision",
    ]

    rows = []

    for full_name, added_term, construct in comparisons:
        keep = performance_df[
            performance_df["Model"].isin([reduced_name, full_name])
        ].copy()

        if keep.empty:
            continue

        for metric in metrics:
            if metric not in keep.columns:
                continue

            reduced_hit = keep.loc[keep["Model"] == reduced_name, metric]
            full_hit = keep.loc[keep["Model"] == full_name, metric]

            if len(reduced_hit) == 0 or len(full_hit) == 0:
                continue

            reduced_value = float(reduced_hit.iloc[0])
            full_value = float(full_hit.iloc[0])

            rows.append({
                "Scenario": scenario_name,
                "Reduced_Model": reduced_name,
                "Full_Model": full_name,
                "Construct": construct,
                "Added_Term": added_term,
                "Metric": metric,
                "M3_controls_value": reduced_value,
                "Adjusted_model_value": full_value,
                "Delta_adjusted_minus_M3": full_value - reduced_value,
            })

    out = pd.DataFrame(rows)

    out.to_csv(
        os.path.join(
            OUTPUT_DIR,
            scenario_name,
            f"dr_incremental_predictive_M3_vs_M4_M5_{scenario_name}.csv",
        ),
        index=False,
    )

    # Convenient separate files.
    if not out.empty:
        out[out["Full_Model"] == "M4_ratio_adjusted"].to_csv(
            os.path.join(
                OUTPUT_DIR,
                scenario_name,
                f"dr_incremental_predictive_M3_vs_M4_{scenario_name}.csv",
            ),
            index=False,
        )
        out[out["Full_Model"] == "M5_presence_adjusted"].to_csv(
            os.path.join(
                OUTPUT_DIR,
                scenario_name,
                f"dr_incremental_predictive_M3_vs_M5_{scenario_name}.csv",
            ),
            index=False,
        )

    display(out)
    return out

## 14. SHAP FOR ADJUSTED M4 (LFCR) AND M5 (LFC) DR MODELS

In [ ]:
def run_adjusted_shap(data, scenario_name, experiment_dict, model_name):
    """
    Publication-ready SHAP analysis for an adjusted delayed-recognition model.

    The model is always fitted with the complete adjusted specification.
    Publication-facing figures intentionally display only the substantive
    predictors, while journal fixed effects and the full-data availability
    indicator remain part of the fitted model and the complete SHAP CSV.

    Used for:
      M4_ratio_adjusted     -> LFCR intensity
      M5_presence_adjusted  -> binary LFC presence
    """
    if not HAS_SHAP:
        return pd.DataFrame()

    scenario_dir = os.path.join(OUTPUT_DIR, scenario_name)
    os.makedirs(scenario_dir, exist_ok=True)

    if model_name not in experiment_dict:
        print(f"Skipping SHAP: {model_name} not defined for {scenario_name}")
        return pd.DataFrame()

    short_name = "M4" if model_name == "M4_ratio_adjusted" else "M5"
    construct = "LFCR" if model_name == "M4_ratio_adjusted" else "LFC"
    plot_features_requested = (
        PLOT_FOCAL_FEATURES_M4
        if model_name == "M4_ratio_adjusted"
        else PLOT_FOCAL_FEATURES_M5
    )

    features = experiment_dict[model_name]
    model_data = data.dropna(subset=features + ["label"]).copy()

    if len(model_data) < MIN_MODEL_N or model_data["label"].nunique() < 2:
        print(
            f"Skipping SHAP for {scenario_name}/{model_name}: "
            "insufficient data/classes"
        )
        return pd.DataFrame()

    X = build_design_matrix(model_data, features)
    y = model_data["label"].astype(int).copy()
    X, _ = remove_zero_variance_features(X)
    X_model = scale_predictive_full(X)

    model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    model.fit(X_model, y)

    try:
        explainer = shap.Explainer(
            model,
            X_model,
            feature_names=X_model.columns.tolist(),
        )
        shap_values = explainer(X_model)

        # Complete importance table: retains every fitted feature for auditability.
        importance = np.abs(shap_values.values).mean(axis=0)
        imp_df = pd.DataFrame({
            "Scenario": scenario_name,
            "Model": model_name,
            "Construct": construct,
            "Feature": X_model.columns,
            "MeanAbsSHAP": importance,
        }).sort_values("MeanAbsSHAP", ascending=False)

        imp_df.to_csv(
            os.path.join(
                scenario_dir,
                f"dr_shap_importance_{short_name}_{scenario_name}.csv",
            ),
            index=False,
        )

        # Main-paper SHAP figures: substantive predictors only.
        plot_features = [
            col for col in plot_features_requested if col in X_model.columns
        ]
        if not plot_features:
            raise ValueError(
                f"No focal SHAP plotting features found for {scenario_name}/{model_name}."
            )

        plot_indices = [X_model.columns.get_loc(col) for col in plot_features]
        plot_X = X_model[plot_features].copy()
        plot_shap = shap_values.values[:, plot_indices]
        plot_feature_names = [clean_feature_name(col) for col in plot_features]

        shap.summary_plot(
            plot_shap,
            plot_X,
            feature_names=plot_feature_names,
            show=False,
            plot_size=FIGSIZE_WIDE,
            max_display=len(plot_features),
        )
        fig = plt.gcf()
        ax = plt.gca()
        ax.set_xlim(*SHAP_SUMMARY_XLIM)
        ax.set_xticks(SHAP_SUMMARY_XTICKS)
        ax.set_xlabel(f"SHAP value ({short_name})")
        apply_figure_fontsize(fig)
        fig.tight_layout()
        save_figure(
            os.path.join(
                scenario_dir,
                f"dr_shap_summary_{short_name}_{scenario_name}",
            )
        )
        plt.show()

        # Parallel Mean |SHAP| bar plot, styled exactly like script 06.
        bar_df = imp_df.loc[imp_df["Feature"].isin(plot_features)].copy()
        bar_df["Feature_Label"] = bar_df["Feature"].apply(clean_feature_name)

        fig, ax = plt.subplots(figsize=FIGSIZE_MAIN)
        sns.barplot(
            data=bar_df,
            x="MeanAbsSHAP",
            y="Feature_Label",
            ax=ax,
            color=BAR_COLOR,
            saturation=1,
            width=0.7,
        )
        ax.set_xlabel(f"Mean |SHAP value| ({short_name})")
        ax.set_xlim(*SHAP_BAR_XLIM)
        ax.set_xticks(SHAP_BAR_XTICKS)
        ax.set_ylabel("")
        ax.grid(axis="x", alpha=0.25)
        ax.grid(axis="y", visible=False)
        apply_figure_fontsize(fig)
        fig.tight_layout()
        save_figure(
            os.path.join(
                scenario_dir,
                f"dr_shap_barplot_{short_name}_{scenario_name}",
            )
        )
        plt.show()

        display(imp_df)
        return imp_df

    except Exception as e:
        print(f"SHAP failed for {scenario_name}/{model_name}: {e}")
        return pd.DataFrame()

## 15. A4 — OUT-OF-FOLD EARLY-IMPACT PREDICTIONS + LATER FOLLOW-UP (M3/M4/M5)

In [ ]:
def make_oof_probabilities(model_data, features, target_col):
    """
    Produce out-of-fold probabilities for every usable paper.

    OOF is intentionally used here instead of in-sample fitted probabilities so that
    the exploratory Actual_3yr=Low / Predicted_3yr=High follow-up is not driven by
    training-set optimism.
    """
    model_data = model_data.dropna(subset=features + [target_col]).copy().reset_index(drop=True)

    if len(model_data) < MIN_MODEL_N or model_data[target_col].nunique() < 2:
        return model_data, None

    y = model_data[target_col].astype(int).reset_index(drop=True)
    min_class_n = int(y.value_counts().min())
    n_splits = min(OOF_MAX_SPLITS, min_class_n)
    if n_splits < 2:
        return model_data, None

    X = build_design_matrix(model_data, features)
    X, _ = remove_zero_variance_features(X)
    X = X.reset_index(drop=True)

    oof_prob = np.full(len(model_data), np.nan, dtype=float)

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    for train_idx, test_idx in cv.split(X, y):
        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_train = y.iloc[train_idx].copy()

        X_train, train_zero = remove_zero_variance_features(X_train)
        X_test = X_test[X_train.columns].copy()

        X_train_model, X_test_model = scale_predictive_train_test(X_train, X_test)

        model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
        model.fit(X_train_model, y_train)
        oof_prob[test_idx] = model.predict_proba(X_test_model)[:, 1]

    return model_data, oof_prob


def run_early_prediction_followup(source_data, scenario_name, experiment_dict):
    follow_dir = os.path.join(OUTPUT_DIR, "early_prediction_followup", scenario_name)
    os.makedirs(follow_dir, exist_ok=True)

    # Early-impact task uses the ORIGINAL early 3-year low/high labels.
    early_data = source_data[source_data["citation_bin"].isin(["low", "high"])].copy()
    early_data["early_label"] = (early_data["citation_bin"] == "high").astype(int)

    reduced_name = (
        "M3_controls_only"
        if scenario_name == "valid_only"
        else "M3_controls_plus_availability"
    )

    # IMPORTANT LEAKAGE SAFEGUARD:
    # The original early citation_bin is derived from the 3-year citation count.
    # `citations` is therefore a valid BASELINE CONTROL for A2/A3 delayed-recognition
    # models, but it must NOT be used to predict the early citation_bin in A4.
    def a4_leakage_safe_features(model_name):
        return [f for f in experiment_dict[model_name] if f != "citations"]

    model_specs = {
        reduced_name: a4_leakage_safe_features(reduced_name),
        "M4_ratio_adjusted": a4_leakage_safe_features("M4_ratio_adjusted"),
        "M5_presence_adjusted": a4_leakage_safe_features("M5_presence_adjusted"),
    }

    summary_rows = []

    for model_name, features in model_specs.items():
        print(f"\nA4 OOF early-impact model: {scenario_name} / {model_name}")

        model_data, oof_prob = make_oof_probabilities(
            early_data,
            features,
            target_col="early_label",
        )

        if oof_prob is None:
            print("Skipping: insufficient data/classes for OOF prediction.")
            continue

        model_data = model_data.copy()
        model_data["oof_p_early_high"] = oof_prob
        model_data["oof_predicted_early_high"] = (model_data["oof_p_early_high"] >= 0.5).astype(int)

        # OOF performance on the original early-impact classification task.
        early_metrics = compute_metrics(
            model_data["early_label"],
            model_data["oof_predicted_early_high"],
            model_data["oof_p_early_high"],
        )

        # Follow-up question is asked ONLY among papers that were actually early Low,
        # and only where later impact is unambiguously Low or High.
        follow = model_data[
            (model_data["early_label"] == 0)
            & (model_data["later_citation_bin"].isin(["low", "high"]))
        ].copy()
        follow["later_high_label"] = (follow["later_citation_bin"] == "high").astype(int)

        if len(follow) == 0 or follow["later_high_label"].nunique() < 2:
            print("Skipping follow-up statistics: no usable early-Low follow-up class variation.")
            continue

        baseline_rate = float(follow["later_high_label"].mean())

        pred_high = follow[follow["oof_predicted_early_high"] == 1]
        pred_low = follow[follow["oof_predicted_early_high"] == 0]

        ph_lh = int(pred_high["later_high_label"].sum())
        ph_ll = int(len(pred_high) - ph_lh)
        pl_lh = int(pred_low["later_high_label"].sum())
        pl_ll = int(len(pred_low) - pl_lh)

        if len(pred_high) > 0 and len(pred_low) > 0:
            odds_ratio, fisher_p = fisher_exact(
                [[ph_lh, ph_ll], [pl_lh, pl_ll]],
                alternative="two-sided",
            )
        else:
            odds_ratio, fisher_p = np.nan, np.nan

        rate_pred_high = float(pred_high["later_high_label"].mean()) if len(pred_high) else np.nan
        rate_pred_low = float(pred_low["later_high_label"].mean()) if len(pred_low) else np.nan

        try:
            delayed_followup_auc = float(
                roc_auc_score(follow["later_high_label"], follow["oof_p_early_high"])
            )
        except Exception:
            delayed_followup_auc = np.nan

        summary = {
            "Scenario": scenario_name,
            "Early_Model": model_name,
            "Features_used_no_early_citation_leakage": ", ".join(features),
            "OOF_N_early_low_high": len(model_data),
            "OOF_early_Accuracy": early_metrics.get("Accuracy", np.nan),
            "OOF_early_Balanced_Accuracy": early_metrics.get("Balanced_Accuracy", np.nan),
            "OOF_early_F1": early_metrics.get("F1", np.nan),
            "OOF_early_ROC_AUC": early_metrics.get("ROC_AUC", np.nan),
            "Followup_N_actual_early_low": len(follow),
            "Followup_LaterHigh_N": int(follow["later_high_label"].sum()),
            "Followup_Baseline_LaterHigh_Rate": baseline_rate,
            "PredictedHigh_N": len(pred_high),
            "PredictedHigh_LaterHigh_N": ph_lh,
            "PredictedHigh_LaterHigh_Rate": rate_pred_high,
            "PredictedLow_N": len(pred_low),
            "PredictedLow_LaterHigh_N": pl_lh,
            "PredictedLow_LaterHigh_Rate": rate_pred_low,
            "Risk_Difference_PredHigh_minus_PredLow": (
                rate_pred_high - rate_pred_low
                if np.isfinite(rate_pred_high) and np.isfinite(rate_pred_low)
                else np.nan
            ),
            "Enrichment_vs_earlyLow_baseline": (
                rate_pred_high / baseline_rate
                if np.isfinite(rate_pred_high) and baseline_rate > 0
                else np.nan
            ),
            "Fisher_Odds_Ratio": odds_ratio,
            "Fisher_p-value": fisher_p,
            "ROC_AUC_oofEarlyHighScore_for_LaterHigh_among_EarlyLow": delayed_followup_auc,
        }
        summary_rows.append(summary)

        # Save all early-low follow-up rows for auditability.
        follow_cols = [
            c for c in [
                "filename",
                "openalex_id",
                "doi",
                "title",
                "journal",
                "journal_short",
                "year",
                "citations",
                "total_citations",
                "post_3yr_citations",
                "citation_bin",
                "later_citation_bin",
                "impact_transition",
                "less_frequent_citation_ratio",
                "has_valid_citation_data",
                "has_less_frequent_citation",
                "references",
                "oof_p_early_high",
                "oof_predicted_early_high",
                "later_high_label",
            ]
            if c in follow.columns
        ]

        follow[follow_cols].sort_values(
            "oof_p_early_high",
            ascending=False,
        ).to_csv(
            os.path.join(
                follow_dir,
                f"early_low_followup_all_{model_name}.csv",
            ),
            index=False,
        )

        # The most interesting candidate set: Actual early Low, OOF Predicted High.
        candidates = follow[follow["oof_predicted_early_high"] == 1].copy()
        candidates[follow_cols].sort_values(
            "oof_p_early_high",
            ascending=False,
        ).to_csv(
            os.path.join(
                follow_dir,
                f"actualLow_predictedHigh_candidates_{model_name}.csv",
            ),
            index=False,
        )

    summary_df = pd.DataFrame(summary_rows)

    # BH/FDR across the three exploratory Fisher tests (M3, M4, M5) within scenario.
    if not summary_df.empty and "Fisher_p-value" in summary_df.columns:
        valid_mask = summary_df["Fisher_p-value"].notna()
        if valid_mask.sum() > 0:
            reject, p_bh, _, _ = multipletests(
                summary_df.loc[valid_mask, "Fisher_p-value"].astype(float),
                alpha=0.05,
                method="fdr_bh",
            )
            summary_df.loc[valid_mask, "Fisher_p_BH"] = p_bh
            summary_df.loc[valid_mask, "Fisher_significant_BH_0.05"] = reject
            summary_df["BH_family"] = "M3, M4, and M5 early-low follow-up Fisher tests within scenario"

    summary_df.to_csv(
        os.path.join(follow_dir, f"early_prediction_followup_summary_{scenario_name}.csv"),
        index=False,
    )
    display(summary_df)
    return summary_df

## 16. RUN A2 / A3 / SHAP (LFCR + LFC)

In [ ]:
performance_valid = run_predictive_analysis(
    df_valid,
    "valid_only",
    experiment_dict_valid,
)
performance_full = run_predictive_analysis(
    df_full,
    "full_dataset",
    experiment_dict_full,
)

inferential_valid = run_inferential_analysis(
    df_valid,
    "valid_only",
    experiment_dict_valid,
)
inferential_full = run_inferential_analysis(
    df_full,
    "full_dataset",
    experiment_dict_full,
)

# A3: both continuous LFCR (M4) and binary LFC presence (M5).
lr_valid = run_incremental_lr_tests(
    df_valid,
    "valid_only",
    experiment_dict_valid,
)
lr_full = run_incremental_lr_tests(
    df_full,
    "full_dataset",
    experiment_dict_full,
)

incremental_predictive_valid = build_incremental_predictive_comparison(
    performance_valid,
    "valid_only",
)
incremental_predictive_full = build_incremental_predictive_comparison(
    performance_full,
    "full_dataset",
)

# SHAP for both adjusted operationalizations.
shap_m4_valid = run_adjusted_shap(
    df_valid,
    "valid_only",
    experiment_dict_valid,
    "M4_ratio_adjusted",
)
shap_m4_full = run_adjusted_shap(
    df_full,
    "full_dataset",
    experiment_dict_full,
    "M4_ratio_adjusted",
)

shap_m5_valid = run_adjusted_shap(
    df_valid,
    "valid_only",
    experiment_dict_valid,
    "M5_presence_adjusted",
)
shap_m5_full = run_adjusted_shap(
    df_full,
    "full_dataset",
    experiment_dict_full,
    "M5_presence_adjusted",
)

# Combined outputs for easier paper tables.
pd.concat([performance_valid, performance_full], ignore_index=True).to_csv(
    os.path.join(OUTPUT_DIR, "dr_predictive_performance_VALID_AND_FULL.csv"),
    index=False,
)

pd.concat([inferential_valid, inferential_full], ignore_index=True).to_csv(
    os.path.join(OUTPUT_DIR, "dr_logistic_inferential_VALID_AND_FULL.csv"),
    index=False,
)

lr_combined = pd.concat([lr_valid, lr_full], ignore_index=True)
lr_combined.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "dr_incremental_LR_M3_vs_M4_M5_VALID_AND_FULL.csv",
    ),
    index=False,
)

# Compatibility / convenient separate combined files.
if not lr_combined.empty:
    lr_combined[
        lr_combined["Full_Model"] == "M4_ratio_adjusted"
    ].to_csv(
        os.path.join(
            OUTPUT_DIR,
            "dr_incremental_LR_M3_vs_M4_VALID_AND_FULL.csv",
        ),
        index=False,
    )
    lr_combined[
        lr_combined["Full_Model"] == "M5_presence_adjusted"
    ].to_csv(
        os.path.join(
            OUTPUT_DIR,
            "dr_incremental_LR_M3_vs_M5_VALID_AND_FULL.csv",
        ),
        index=False,
    )

incremental_predictive_combined = pd.concat(
    [incremental_predictive_valid, incremental_predictive_full],
    ignore_index=True,
)
incremental_predictive_combined.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "dr_incremental_predictive_M3_vs_M4_M5_VALID_AND_FULL.csv",
    ),
    index=False,
)

if not incremental_predictive_combined.empty:
    incremental_predictive_combined[
        incremental_predictive_combined["Full_Model"] == "M4_ratio_adjusted"
    ].to_csv(
        os.path.join(
            OUTPUT_DIR,
            "dr_incremental_predictive_M3_vs_M4_VALID_AND_FULL.csv",
        ),
        index=False,
    )
    incremental_predictive_combined[
        incremental_predictive_combined["Full_Model"] == "M5_presence_adjusted"
    ].to_csv(
        os.path.join(
            OUTPUT_DIR,
            "dr_incremental_predictive_M3_vs_M5_VALID_AND_FULL.csv",
        ),
        index=False,
    )

shap_combined = pd.concat(
    [shap_m4_valid, shap_m4_full, shap_m5_valid, shap_m5_full],
    ignore_index=True,
)
if not shap_combined.empty:
    shap_combined.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "dr_shap_importance_M4_M5_VALID_AND_FULL.csv",
        ),
        index=False,
    )

## 17. RUN A4 — EARLY-PREDICTION FOLLOW-UP

In [ ]:
# IMPORTANT: A4 needs the full selected-milestone dataset, not dr_all, because it
# trains the original early Low-vs-High impact task first and only then inspects
# later outcomes among Actual early-Low papers.

early_source_valid = df[df["has_valid_citation_data"] == 1].copy()
early_source_full = df.copy()
early_source_full["less_frequent_citation_ratio"] = early_source_full[
    "less_frequent_citation_ratio"
].fillna(0)

followup_valid = run_early_prediction_followup(
    early_source_valid,
    "valid_only",
    experiment_dict_valid,
)

followup_full = run_early_prediction_followup(
    early_source_full,
    "full_dataset",
    experiment_dict_full,
)

followup_combined = pd.concat(
    [followup_valid, followup_full],
    ignore_index=True,
)
followup_combined.to_csv(
    os.path.join(OUTPUT_DIR, "early_prediction_followup_summary_VALID_AND_FULL.csv"),
    index=False,
)

## 18. A5 — ROBUSTNESS / SENSITIVITY ANALYSES

Two sensitivity checks complement the primary `Low_3yr -> High_later` analysis:

1. **Post-3-year citation-gain sensitivity.**  Among papers that were early Low,
   the outcome is High vs Low based only on citations accumulated *after* the
   3-year window (`post_3yr_citations`), using the same journal-year / journal-period
   rank-tertile structure. This removes the mechanical contribution of early
   citations from the later-impact quantity.
2. **Recent-cohort exclusion sensitivity.**  The most recent publication year in
   the selected milestone is excluded, giving the remaining papers a longer
   post-3-year observation window.

Both checks retain the exact early 3-year citation count as a baseline control in
M3/M4/M5. They are robustness analyses, not replacements for the primary outcome.

In [ ]:
def run_adjusted_sensitivity(
    data,
    scenario_name,
    experiment_dict,
    sensitivity_name,
    outcome_description,
):
    """Run adjusted M3/M4/M5 inference and M3->M4/M5 nested LR for a sensitivity outcome."""
    sensitivity_dir = os.path.join(OUTPUT_DIR, "sensitivities", sensitivity_name, scenario_name)
    os.makedirs(sensitivity_dir, exist_ok=True)

    reduced_name = (
        "M3_controls_only"
        if scenario_name == "valid_only"
        else "M3_controls_plus_availability"
    )
    model_names = [reduced_name, "M4_ratio_adjusted", "M5_presence_adjusted"]

    inference_rows = []

    for model_name in model_names:
        features = experiment_dict[model_name]
        model_data = data.dropna(subset=features + ["label"]).copy()

        if len(model_data) < MIN_MODEL_N or model_data["label"].nunique() < 2:
            print(
                f"Sensitivity {sensitivity_name}: skipping {scenario_name}/{model_name} "
                "because of insufficient data/classes"
            )
            continue

        X = build_design_matrix(model_data, features)
        y = model_data["label"].astype(int).copy()
        X, _ = remove_zero_variance_features(X)
        if X.shape[1] == 0:
            continue

        if "year" in X.columns:
            X["year"] = X["year"] - X["year"].mean()

        try:
            fitted = sm.Logit(
                y,
                sm.add_constant(X, has_constant="add"),
            ).fit(disp=0)
            conf = fitted.conf_int()

            for feature in fitted.params.index:
                beta = float(fitted.params[feature])
                ci_low = float(conf.loc[feature, 0])
                ci_high = float(conf.loc[feature, 1])

                row = {
                    "Sensitivity": sensitivity_name,
                    "Outcome": outcome_description,
                    "Scenario": scenario_name,
                    "Model": model_name,
                    "Feature": feature,
                    "N": len(model_data),
                    "Positive_N": int(y.sum()),
                    "Positive_Rate": float(y.mean()),
                    "Coefficient": beta,
                    "Std_Error": float(fitted.bse[feature]),
                    "Odds_Ratio": float(np.exp(beta)),
                    "OR_CI95_Lower": float(np.exp(ci_low)),
                    "OR_CI95_Upper": float(np.exp(ci_high)),
                    "p-value": float(fitted.pvalues[feature]),
                    "Pseudo_R2": float(fitted.prsquared),
                    "AIC": float(fitted.aic),
                    "BIC": float(fitted.bic),
                }

                if feature == "less_frequent_citation_ratio":
                    row["OR_per_0.10_ratio_increase"] = float(np.exp(beta * 0.10))
                    row["OR_per_0.10_CI95_Lower"] = float(np.exp(ci_low * 0.10))
                    row["OR_per_0.10_CI95_Upper"] = float(np.exp(ci_high * 0.10))

                inference_rows.append(row)

        except Exception as e:
            print(
                f"Sensitivity inference failed for {sensitivity_name}/"
                f"{scenario_name}/{model_name}: {e}"
            )

    inference_df = pd.DataFrame(inference_rows)
    inference_df.to_csv(
        os.path.join(sensitivity_dir, "adjusted_inferential_M3_M4_M5.csv"),
        index=False,
    )

    # Nested incremental tests; each reduced/full comparison uses exactly the
    # rows required by its own full model.
    lr_rows = []
    for full_name, added_term, construct in [
        ("M4_ratio_adjusted", "less_frequent_citation_ratio", "LFCR"),
        ("M5_presence_adjusted", "has_less_frequent_citation", "LFC"),
    ]:
        reduced_features = experiment_dict[reduced_name]
        full_features = experiment_dict[full_name]
        model_data = data.dropna(subset=full_features + ["label"]).copy()

        if len(model_data) < MIN_MODEL_N or model_data["label"].nunique() < 2:
            continue

        y = model_data["label"].astype(int).copy()
        X_reduced = build_design_matrix(model_data, reduced_features)
        X_full = build_design_matrix(model_data, full_features)
        X_reduced, _ = remove_zero_variance_features(X_reduced)
        X_full, _ = remove_zero_variance_features(X_full)

        if "year" in X_reduced.columns:
            year_mean = X_reduced["year"].mean()
            X_reduced["year"] = X_reduced["year"] - year_mean
            if "year" in X_full.columns:
                X_full["year"] = X_full["year"] - year_mean

        try:
            reduced_fit = sm.Logit(
                y,
                sm.add_constant(X_reduced, has_constant="add"),
            ).fit(disp=0)
            full_fit = sm.Logit(
                y,
                sm.add_constant(X_full, has_constant="add"),
            ).fit(disp=0)

            lr_stat = 2.0 * (full_fit.llf - reduced_fit.llf)
            df_diff = int(full_fit.df_model - reduced_fit.df_model)
            p_value = float(chi2.sf(lr_stat, df_diff)) if df_diff > 0 else np.nan

            lr_rows.append({
                "Sensitivity": sensitivity_name,
                "Outcome": outcome_description,
                "Scenario": scenario_name,
                "Reduced_Model": reduced_name,
                "Full_Model": full_name,
                "Construct": construct,
                "Added_Term": added_term,
                "N": len(model_data),
                "Positive_N": int(y.sum()),
                "LR_statistic": float(lr_stat),
                "df": df_diff,
                "p-value": p_value,
                "Reduced_Pseudo_R2": float(reduced_fit.prsquared),
                "Full_Pseudo_R2": float(full_fit.prsquared),
                "Delta_Pseudo_R2": float(full_fit.prsquared - reduced_fit.prsquared),
                "Reduced_AIC": float(reduced_fit.aic),
                "Full_AIC": float(full_fit.aic),
                "Delta_AIC_Full_minus_Reduced": float(full_fit.aic - reduced_fit.aic),
            })
        except Exception as e:
            print(
                f"Sensitivity LR failed for {sensitivity_name}/{scenario_name}/"
                f"{reduced_name} vs {full_name}: {e}"
            )

    lr_df = pd.DataFrame(lr_rows)
    if not lr_df.empty and "p-value" in lr_df.columns:
        valid_mask = lr_df["p-value"].notna()
        if valid_mask.sum() > 0:
            reject, p_bh, _, _ = multipletests(
                lr_df.loc[valid_mask, "p-value"].astype(float),
                alpha=0.05,
                method="fdr_bh",
            )
            lr_df.loc[valid_mask, "p-value_BH"] = p_bh
            lr_df.loc[valid_mask, "significant_BH_0.05"] = reject
            lr_df["BH_family"] = (
                f"M3->M4 LFCR and M3->M5 LFC within {sensitivity_name}/{scenario_name}"
            )

    lr_df.to_csv(
        os.path.join(sensitivity_dir, "incremental_LR_M3_vs_M4_M5.csv"),
        index=False,
    )

    print(f"\nA5 sensitivity: {sensitivity_name} / {scenario_name}")
    if not inference_df.empty:
        display(inference_df)
    if not lr_df.empty:
        display(lr_df)

    return inference_df, lr_df


# ------------------------------------------------------------------
# A5a. Post-3-year citation-gain sensitivity
# ------------------------------------------------------------------
postgain_all = df[
    (df["citation_bin"] == "low")
    & (df["post_3yr_gain_bin"].isin(["low", "high"]))
].copy()
postgain_all["label"] = (postgain_all["post_3yr_gain_bin"] == "high").astype(int)
postgain_all["post_3yr_gain_sensitivity_label"] = postgain_all["label"]

postgain_counts = pd.crosstab(
    postgain_all["citation_bin"],
    postgain_all["post_3yr_gain_bin"],
    dropna=False,
)
postgain_counts.to_csv(
    os.path.join(OUTPUT_DIR, "sensitivity_post3yr_gain_transition_counts.csv")
)

postgain_valid = postgain_all[postgain_all["has_valid_citation_data"] == 1].copy()
postgain_full = postgain_all.copy()
postgain_full["less_frequent_citation_ratio"] = postgain_full[
    "less_frequent_citation_ratio"
].fillna(0)

postgain_inf_valid, postgain_lr_valid = run_adjusted_sensitivity(
    postgain_valid,
    "valid_only",
    experiment_dict_valid,
    "post_3yr_gain",
    "early Low -> high post-3yr citation gain vs early Low -> low post-3yr citation gain",
)
postgain_inf_full, postgain_lr_full = run_adjusted_sensitivity(
    postgain_full,
    "full_dataset",
    experiment_dict_full,
    "post_3yr_gain",
    "early Low -> high post-3yr citation gain vs early Low -> low post-3yr citation gain",
)

pd.concat([postgain_inf_valid, postgain_inf_full], ignore_index=True).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "sensitivity_post3yr_gain_inferential_VALID_AND_FULL.csv",
    ),
    index=False,
)
pd.concat([postgain_lr_valid, postgain_lr_full], ignore_index=True).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "sensitivity_post3yr_gain_LR_VALID_AND_FULL.csv",
    ),
    index=False,
)


# ------------------------------------------------------------------
# A5b. Exclude the most recent publication cohort
# ------------------------------------------------------------------
_recent_year_values = pd.to_numeric(dr_all["year"], errors="coerce").dropna()
MOST_RECENT_YEAR_EXCLUDED = (
    int(_recent_year_values.max()) if len(_recent_year_values) else None
)

if MOST_RECENT_YEAR_EXCLUDED is not None:
    recent_excluded_all = dr_all[
        pd.to_numeric(dr_all["year"], errors="coerce") < MOST_RECENT_YEAR_EXCLUDED
    ].copy()
else:
    recent_excluded_all = dr_all.iloc[0:0].copy()

recent_excluded_valid = recent_excluded_all[
    recent_excluded_all["has_valid_citation_data"] == 1
].copy()
recent_excluded_full = recent_excluded_all.copy()
recent_excluded_full["less_frequent_citation_ratio"] = recent_excluded_full[
    "less_frequent_citation_ratio"
].fillna(0)

recent_inf_valid, recent_lr_valid = run_adjusted_sensitivity(
    recent_excluded_valid,
    "valid_only",
    experiment_dict_valid,
    "exclude_most_recent_year",
    f"primary delayed-recognition outcome excluding publication year {MOST_RECENT_YEAR_EXCLUDED}",
)
recent_inf_full, recent_lr_full = run_adjusted_sensitivity(
    recent_excluded_full,
    "full_dataset",
    experiment_dict_full,
    "exclude_most_recent_year",
    f"primary delayed-recognition outcome excluding publication year {MOST_RECENT_YEAR_EXCLUDED}",
)

pd.concat([recent_inf_valid, recent_inf_full], ignore_index=True).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "sensitivity_exclude_most_recent_year_inferential_VALID_AND_FULL.csv",
    ),
    index=False,
)
pd.concat([recent_lr_valid, recent_lr_full], ignore_index=True).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "sensitivity_exclude_most_recent_year_LR_VALID_AND_FULL.csv",
    ),
    index=False,
)

## 19. METADATA / CONFIG

In [ ]:
metadata_07 = {
    "milestone": MILESTONE,
    "current_milestone_order": CURRENT_MILESTONE_ORDER,
    "analysis_name": "delayed_recognition",
    "milestone_filtering": "cumulative_eligible_dataset",
    "milestone_consistency_with_06": bool(consistency_passed),
    "n_cumulative_eligible": int(len(df)),
    "n_with_later_bin": int(df["later_citation_bin"].notna().sum()),
    "n_with_post_3yr_gain_bin": int(df["post_3yr_gain_bin"].notna().sum()),
    "n_early_low": int((df["citation_bin"] == "low").sum()),
    "n_early_low_later_low_high": int(len(dr_all)),
    "n_delayed_recognition_low_to_high": int(dr_all["label"].sum()),
    "delayed_recognition_rate_among_binary_early_low": (
        float(dr_all["label"].mean()) if len(dr_all) else np.nan
    ),
    "n_valid_dr": int(len(df_valid)),
    "n_full_dr": int(len(df_full)),
    "lfcr_coverage_dr_population": (
        float(dr_all["has_valid_citation_data"].mean()) if len(dr_all) else np.nan
    ),
    "lfc_presence_rate_dr_population": (
        float(dr_all["has_less_frequent_citation"].mean()) if len(dr_all) else np.nan
    ),
    "baseline_3yr_citation_control_in_adjusted_DR_models": True,
    "n_post3yr_gain_sensitivity_population": int(len(postgain_all)),
    "most_recent_year_excluded_sensitivity": MOST_RECENT_YEAR_EXCLUDED,
    "n_recent_year_excluded_sensitivity_population": int(len(recent_excluded_all)),
    "later_bin_method": "rank(method=first) then qcut(q=3)",
    "later_bin_value": "total_citations",
    "post3yr_gain_bin_value": "post_3yr_citations = max(total_citations - citations, 0)",
    "later_bin_cohort": "reuse original binning_level: journal_year or journal_period",
    "strict_sleeping_beauty_claim": False,
}

pd.Series(metadata_07).to_csv(
    os.path.join(OUTPUT_DIR, "metadata_07.csv")
)

config_07 = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "oof_max_splits": OOF_MAX_SPLITS,
    "focal_delayed_models": "M4_ratio_adjusted (LFCR intensity); M5_presence_adjusted (LFC presence)",
    "focal_delayed_predictors": "less_frequent_citation_ratio (LFCR); has_less_frequent_citation (LFC presence)",
    "adjusted_baseline_controls_valid": "citations + references + year + journal",
    "adjusted_baseline_controls_full": "has_valid_citation_data + citations + references + year + journal",
    "baseline_3yr_citation_control_reason": (
        "Controls exact citation level at end of early 3-year window so LFCR/LFC are tested beyond early citation standing"
    ),
    "primary_delayed_outcome": "early low -> later high vs early low -> later low",
    "later_mid_policy": "excluded from binary delayed-recognition models",
    "A3_incremental_tests": "M3 vs M4 (LFCR); M3 vs M5 (LFC)",
    "A3_multiple_testing": "BH/FDR across M3->M4 and M3->M5 within scenario",
    "A4_probability_threshold": 0.5,
    "A4_prediction_method": "out-of-fold StratifiedKFold",
    "A4_models": "M3 controls vs M4 controls+LFCR vs M5 controls+LFC",
    "A4_early_citation_leakage_policy": (
        "citations excluded from A4 predictors because early citation_bin is derived from the 3-year citation count"
    ),
    "A4_multiple_testing": "BH/FDR across M3, M4, M5 Fisher tests within scenario",
    "SHAP_models": "M4_ratio_adjusted and M5_presence_adjusted",
    "A5_sensitivity_post3yr_gain": (
        "early Low -> high vs low post-3yr citation-gain tertile; mid excluded"
    ),
    "A5_sensitivity_recent_cohort": (
        "repeat primary adjusted inference after excluding the most recent publication year"
    ),
    "source_clean_snapshot": CLEAN_DATASET_PATH,
    "source_rare_snapshot": RARE_DATASET_PATH,
    "source_sampling_plan_snapshot": SAMPLING_PLAN_PATH,
}

pd.Series(config_07).to_csv(
    os.path.join(OUTPUT_DIR, "config_07.csv")
)

## 20. AUTOMATIC COMPARISON ACROSS COMPLETED 07 MILESTONES

In [ ]:
comparison_rows = []

for milestone_name in sorted(existing_dirs, key=infer_milestone_order):
    delayed_dir = os.path.join(
        MILESTONES_BASE_DIR,
        milestone_name,
        "outputs",
        "delayed_recognition",
    )
    metadata_path = os.path.join(delayed_dir, "metadata_07.csv")
    inference_path = os.path.join(
        delayed_dir,
        "dr_logistic_inferential_VALID_AND_FULL.csv",
    )

    # Prefer the new LFCR+LFC combined LR file; fall back to the older LFCR-only
    # filename so previously completed 07 milestones remain readable.
    lr_path_new = os.path.join(
        delayed_dir,
        "dr_incremental_LR_M3_vs_M4_M5_VALID_AND_FULL.csv",
    )
    lr_path_old = os.path.join(
        delayed_dir,
        "dr_incremental_LR_M3_vs_M4_VALID_AND_FULL.csv",
    )
    lr_path = lr_path_new if os.path.exists(lr_path_new) else lr_path_old

    perf_path = os.path.join(
        delayed_dir,
        "dr_predictive_performance_VALID_AND_FULL.csv",
    )
    follow_path = os.path.join(
        delayed_dir,
        "early_prediction_followup_summary_VALID_AND_FULL.csv",
    )

    if not os.path.exists(metadata_path):
        continue

    metadata = pd.read_csv(
        metadata_path,
        index_col=0,
        header=None,
    ).squeeze("columns").to_dict()

    row = {
        "milestone": milestone_name,
        "current_milestone_order": metadata.get("current_milestone_order"),
        "n_cumulative_eligible": metadata.get("n_cumulative_eligible"),
        "n_early_low_later_low_high": metadata.get("n_early_low_later_low_high"),
        "n_delayed_recognition_low_to_high": metadata.get(
            "n_delayed_recognition_low_to_high"
        ),
        "delayed_recognition_rate": metadata.get(
            "delayed_recognition_rate_among_binary_early_low"
        ),
        "lfcr_coverage_dr_population": metadata.get("lfcr_coverage_dr_population"),
        "lfc_presence_rate_dr_population": metadata.get(
            "lfc_presence_rate_dr_population"
        ),
        "baseline_3yr_citation_control_in_adjusted_DR_models": metadata.get(
            "baseline_3yr_citation_control_in_adjusted_DR_models"
        ),
        "most_recent_year_excluded_sensitivity": metadata.get(
            "most_recent_year_excluded_sensitivity"
        ),
        "n_post3yr_gain_sensitivity_population": metadata.get(
            "n_post3yr_gain_sensitivity_population"
        ),
        "n_recent_year_excluded_sensitivity_population": metadata.get(
            "n_recent_year_excluded_sensitivity_population"
        ),
    }

    # M4 LFCR and M5 LFC inferential estimates.
    if os.path.exists(inference_path):
        inf = pd.read_csv(inference_path)

        for scenario in ["valid_only", "full_dataset"]:
            # Continuous LFCR: report OR per +0.10.
            hit_m4 = inf[
                (inf["Scenario"] == scenario)
                & (inf["Model"] == "M4_ratio_adjusted")
                & (inf["Feature"] == "less_frequent_citation_ratio")
            ]
            if len(hit_m4):
                prefix = f"{scenario}_M4_LFCR"
                row[f"{prefix}_coef"] = hit_m4["Coefficient"].iloc[0]
                row[f"{prefix}_OR_per_0.10"] = hit_m4.get(
                    "OR_per_0.10_ratio_increase",
                    pd.Series([np.nan]),
                ).iloc[0]
                row[f"{prefix}_OR_per_0.10_CI95_Lower"] = hit_m4.get(
                    "OR_per_0.10_CI95_Lower",
                    pd.Series([np.nan]),
                ).iloc[0]
                row[f"{prefix}_OR_per_0.10_CI95_Upper"] = hit_m4.get(
                    "OR_per_0.10_CI95_Upper",
                    pd.Series([np.nan]),
                ).iloc[0]
                row[f"{prefix}_pvalue"] = hit_m4["p-value"].iloc[0]

            # Binary LFC presence: report ordinary OR and 95% CI.
            hit_m5 = inf[
                (inf["Scenario"] == scenario)
                & (inf["Model"] == "M5_presence_adjusted")
                & (inf["Feature"] == "has_less_frequent_citation")
            ]
            if len(hit_m5):
                prefix = f"{scenario}_M5_LFC"
                row[f"{prefix}_coef"] = hit_m5["Coefficient"].iloc[0]
                row[f"{prefix}_OR"] = hit_m5["Odds_Ratio"].iloc[0]
                row[f"{prefix}_OR_CI95_Lower"] = hit_m5["OR_CI95_Lower"].iloc[0]
                row[f"{prefix}_OR_CI95_Upper"] = hit_m5["OR_CI95_Upper"].iloc[0]
                row[f"{prefix}_pvalue"] = hit_m5["p-value"].iloc[0]

    # A3 incremental LR tests for both alternative operationalizations.
    if os.path.exists(lr_path):
        lr = pd.read_csv(lr_path)
        for scenario in ["valid_only", "full_dataset"]:
            for full_model, short in [
                ("M4_ratio_adjusted", "M3vsM4_LFCR"),
                ("M5_presence_adjusted", "M3vsM5_LFC"),
            ]:
                hit = lr[
                    (lr["Scenario"] == scenario)
                    & (lr["Full_Model"] == full_model)
                ]
                if len(hit):
                    prefix = f"{scenario}_{short}"
                    row[f"{prefix}_LR"] = hit["LR_statistic"].iloc[0]
                    row[f"{prefix}_pvalue"] = hit["p-value"].iloc[0]
                    if "p-value_BH" in hit.columns:
                        row[f"{prefix}_pvalue_BH"] = hit["p-value_BH"].iloc[0]
                    row[f"{prefix}_delta_pseudoR2"] = hit["Delta_Pseudo_R2"].iloc[0]
                    row[f"{prefix}_delta_AIC"] = hit[
                        "Delta_AIC_Full_minus_Reduced"
                    ].iloc[0]

    # Predictive performance for M3, M4, and M5.
    if os.path.exists(perf_path):
        perf = pd.read_csv(perf_path)
        for scenario in ["valid_only", "full_dataset"]:
            for model_name in [
                "M3_controls_only",
                "M3_controls_plus_availability",
                "M4_ratio_adjusted",
                "M5_presence_adjusted",
            ]:
                hit = perf[
                    (perf["Scenario"] == scenario)
                    & (perf["Model"] == model_name)
                ]
                if len(hit):
                    prefix = f"{scenario}_{model_name}"
                    if "ROC_AUC" in hit.columns:
                        row[f"{prefix}_ROC_AUC"] = hit["ROC_AUC"].iloc[0]
                    if "Average_Precision" in hit.columns:
                        row[f"{prefix}_Average_Precision"] = hit[
                            "Average_Precision"
                        ].iloc[0]

    # A4 OOF early-impact follow-up for M3, M4, and M5.
    if os.path.exists(follow_path):
        fol = pd.read_csv(follow_path)
        for scenario in ["valid_only", "full_dataset"]:
            for model_name in [
                "M3_controls_only",
                "M3_controls_plus_availability",
                "M4_ratio_adjusted",
                "M5_presence_adjusted",
            ]:
                hit = fol[
                    (fol["Scenario"] == scenario)
                    & (fol["Early_Model"] == model_name)
                ]
                if len(hit):
                    prefix = f"A4_{scenario}_{model_name}"
                    row[f"{prefix}_predHigh_laterHigh_rate"] = hit[
                        "PredictedHigh_LaterHigh_Rate"
                    ].iloc[0]
                    row[f"{prefix}_predLow_laterHigh_rate"] = hit[
                        "PredictedLow_LaterHigh_Rate"
                    ].iloc[0]
                    row[f"{prefix}_Fisher_OR"] = hit[
                        "Fisher_Odds_Ratio"
                    ].iloc[0]
                    row[f"{prefix}_Fisher_p"] = hit["Fisher_p-value"].iloc[0]
                    if "Fisher_p_BH" in hit.columns:
                        row[f"{prefix}_Fisher_p_BH"] = hit[
                            "Fisher_p_BH"
                        ].iloc[0]
                    row[f"{prefix}_later_followup_AUC"] = hit[
                        "ROC_AUC_oofEarlyHighScore_for_LaterHigh_among_EarlyLow"
                    ].iloc[0]

    comparison_rows.append(row)

milestone_comparison = pd.DataFrame(comparison_rows)
if (
    not milestone_comparison.empty
    and "current_milestone_order" in milestone_comparison.columns
):
    milestone_comparison["current_milestone_order"] = pd.to_numeric(
        milestone_comparison["current_milestone_order"],
        errors="coerce",
    )
    milestone_comparison = milestone_comparison.sort_values(
        "current_milestone_order"
    ).reset_index(drop=True)

comparison_path = os.path.join(
    MILESTONES_BASE_DIR,
    "delayed_recognition_milestone_comparison_summary.csv",
)
milestone_comparison.to_csv(comparison_path, index=False)

print("\n====================================================")
print("07 DELAYED-RECOGNITION ANALYSIS COMPLETED")
print("Milestone:", MILESTONE)
print("Outputs:", OUTPUT_DIR)
print("Cross-milestone summary:", comparison_path)
print("====================================================")

display(pd.DataFrame([metadata_07]))
display(milestone_comparison)

# END